# IR 1 — Vibe-code data examination and data carpentry

This is the workshop notebook. It reads the table produced by `00_IR_Get_Data_NISTChemPy_v2.ipynb`: one standardized `IR` column, one `SMILES` column, and one binary categorical column per functional group. The packaged reference is used only when a fresh crawl has not been completed, so the workshop can be rehearsed without WebBook access.

The prompts are intentionally specific: ask an LLM to inspect before it modifies, make its row-removal decisions visible, and write a new artifact rather than overwrite the source.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs/carpentry')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCRAPER_OUTPUT = Path('outputs/nist_acquisition/IR_SMILES_functional_groups.parquet')
MODEL_READY = OUTPUT_DIR / 'IR_model_ready.parquet'
LABELS = ['Vinylic', 'Arene', 'Alcohol', 'Ether', 'Amine', 'Nitrile', 'Aldehyde', 'Ketone', 'CarboxylAcid', 'Ester', 'Amide', 'AcidChloride']

# The reference release is converted to the exact schema promised by notebook 0.
def packaged_scraper_schema():
    reference = pd.read_parquet(DATA_DIR / 'IR_v09_df_all_features.parquet')
    keep = ['NIST_molecule', 'CAS_NO', 'Name', 'molform', 'smiles', 'x', 'y'] + LABELS
    reference = reference[keep].rename(columns={'smiles': 'SMILES', 'x': 'wavenumber', 'y': 'IR'})
    return reference

source = pd.read_parquet(SCRAPER_OUTPUT) if SCRAPER_OUTPUT.exists() else packaged_scraper_schema()
print(f'Input: {SCRAPER_OUTPUT if SCRAPER_OUTPUT.exists() else "packaged scraper-equivalent reference"}')
print(f'Rows: {len(source):,}; columns: {source.columns.tolist()}')
display(source.head(3))


## Prompt 1 — examine, do not edit

> Inspect this IR dataset without modifying it. Report schema, missingness, duplicate NIST and CAS identifiers, the lengths and finite-value status of the `IR` and `wavenumber` arrays, whether every spectrum shares one grid, label prevalence, and potential leakage into a functional-group classifier. Explain the significance of every check.


In [ ]:
required = {'NIST_molecule', 'CAS_NO', 'SMILES', 'IR', 'wavenumber'} | set(LABELS)
missing = required - set(source.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

def audit_array(values):
    vector = np.asarray(values, dtype=float)
    return len(vector), vector.ndim == 1 and np.isfinite(vector).all()

ir_audit = source.IR.map(audit_array)
grid_audit = source.wavenumber.map(audit_array)
print('Missing values:', source[list(required)].isna().sum().to_dict())
print('Duplicate NIST IDs:', source.duplicated('NIST_molecule').sum())
print('Duplicate CAS numbers:', source.duplicated('CAS_NO').sum())
print('IR length counts:', ir_audit.map(lambda result: result[0]).value_counts().to_dict())
print('Finite IR arrays:', ir_audit.map(lambda result: result[1]).mean())
display((100 * source[LABELS].mean()).rename('positive spectra (%)').sort_values(ascending=False).round(1).to_frame())


## Prompt 2 — create a model-ready artifact

> Create a new model-ready Parquet file without modifying the source. Keep only finite 500-point IR and wavenumber arrays, remove duplicate NIST identifiers and CAS numbers, verify the common grid, retain categorical functional-group labels, and print a before/after row-count audit. Reload the saved file and validate its schema.


In [ ]:
def valid_row(row):
    ir, grid = np.asarray(row.IR, dtype=float), np.asarray(row.wavenumber, dtype=float)
    return ir.ndim == grid.ndim == 1 and len(ir) == len(grid) == 500 and np.isfinite(ir).all() and np.isfinite(grid).all()

starting = len(source)
clean = source[source.apply(valid_row, axis=1)].copy()
after_arrays = len(clean)
clean = clean.dropna(subset=['NIST_molecule', 'CAS_NO', 'SMILES']).drop_duplicates('NIST_molecule').drop_duplicates('CAS_NO')
after_ids = len(clean)
grid = np.asarray(clean.iloc[0].wavenumber, dtype=float)
if not all(np.array_equal(grid, np.asarray(values, dtype=float)) for values in clean.wavenumber):
    raise ValueError('The spectra do not share a common grid; rerun acquisition standardization.')

columns = ['NIST_molecule', 'CAS_NO', 'Name', 'molform', 'SMILES', 'wavenumber', 'IR'] + LABELS
model_ready = clean[columns].copy()
model_ready.to_parquet(MODEL_READY, index=False)
round_trip = pd.read_parquet(MODEL_READY)
if round_trip.columns.tolist() != columns or not round_trip.IR.map(len).eq(500).all():
    raise ValueError('Round-trip validation failed.')
print(pd.DataFrame([{'starting rows': starting, 'after array checks': after_arrays,
                     'after identifier checks': after_ids, 'saved rows': len(round_trip)}]).to_string(index=False))
print(f'Wrote and verified {MODEL_READY}')

fig, ax = plt.subplots(figsize=(7, 4))
for _, row in round_trip.sample(min(12, len(round_trip)), random_state=42).iterrows():
    ax.plot(row.wavenumber, row.IR, alpha=0.35, linewidth=0.8)
ax.invert_xaxis(); ax.set(xlabel='Wavenumber (cm$^{-1}$)', ylabel='Normalized IR signal', title='Carpentered IR spectra')
ax.grid(alpha=0.2); plt.show()


## Handoff

The next notebook reads only `outputs/carpentry/IR_model_ready.parquet`. This makes it clear that the training result is based on a particular, validated data artifact—not on hidden live downloads or unrecorded cleaning.
